# 第 8 章：Hugging Face 工作流

这个 notebook 对应 `lessons/08_huggingface_workflow.md`。为了让课程在离线环境也可验证，这里先演示 Hugging Face 工作流的本地契约：加载计划、tokenizer batch、causal LM labels、chat template fallback、保存清单。真正下载模型时，再把这些契约替换到 `AutoTokenizer`、`AutoModelForCausalLM` 和 `Trainer` 上。

In [ ]:
import json
import tempfile
from pathlib import Path

import torch

from src.finetune.hf_workflow import (
    GenerationSettings,
    HFLoadPlan,
    assert_causal_lm_logits_shape,
    causal_lm_data_collator,
    deterministic_split,
    format_messages_fallback,
    greedy_next_token,
    save_pretrained_artifacts,
    validate_tokenizer_batch,
    verify_pretrained_artifacts,
    write_generation_settings,
)

## 1. Load Plan

加载开源模型时，`model_id`、revision、dtype、device map 和 trust policy 都应显式记录。

In [ ]:
plan = HFLoadPlan(
    model_id="sshleifer/tiny-gpt2",
    revision="main",
    torch_dtype="float16",
    device_map="auto",
)
print(plan.tokenizer_kwargs())
print(plan.model_kwargs())

## 2. Tokenizer Batch 契约

`AutoTokenizer(...)` 的输出至少要包含同 shape 的 `input_ids` 和 `attention_mask`。

In [ ]:
batch = {
    "input_ids": torch.tensor([[1, 2, 3], [4, 5, 0]], dtype=torch.long),
    "attention_mask": torch.tensor([[1, 1, 1], [1, 1, 0]], dtype=torch.long),
}
validate_tokenizer_batch(batch)
print(batch["input_ids"].shape)

## 3. Causal LM Collator

普通续写任务里，labels 是 input_ids 的拷贝，但 padding 位置必须改成 `-100`。

In [ ]:
collated = causal_lm_data_collator(
    [{"input_ids": [5, 6, 7]}, {"input_ids": [8]}],
    pad_token_id=0,
)
print(collated["input_ids"])
print(collated["labels"])

## 4. Logits Shape 与 Greedy Generation

causal LM logits 最后一维必须等于 vocab size；greedy generation 在同一 logits 下应确定。

In [ ]:
logits = torch.randn(2, 3, 11)
assert_causal_lm_logits_shape(logits, collated["input_ids"], expected_vocab_size=11)
next_token = greedy_next_token(logits[:, -1, :])
print(next_token)

## 5. Chat Template Fallback

真实项目应优先使用 tokenizer 自带的 `apply_chat_template`。没有模板时，项目内必须固定 fallback 格式。

In [ ]:
prompt = format_messages_fallback(
    [
        {"role": "system", "content": "你是谨慎的中文技术助教。"},
        {"role": "user", "content": "解释什么是 causal mask。"},
    ],
)
print(prompt)

## 6. Split 与 Save Pretrained 清单

保存模型时要同时保存 model、tokenizer 和 workflow manifest，否则实验不可复现。

In [ ]:
class FakePretrained:
    def __init__(self, filename, payload):
        self.filename = filename
        self.payload = payload

    def save_pretrained(self, output_dir):
        path = Path(output_dir) / self.filename
        path.write_text(json.dumps(self.payload, sort_keys=True))


items = [{"id": str(index)} for index in range(10)]
train, val = deterministic_split(items, val_ratio=0.2, seed=0)
print(len(train), len(val))

with tempfile.TemporaryDirectory() as tmpdir:
    save_pretrained_artifacts(
        tmpdir,
        FakePretrained("config.json", {"model_type": "fake-causal-lm"}),
        FakePretrained("tokenizer_config.json", {"model_max_length": 16}),
        {"model_id": plan.model_id, "dataset_version": "tiny-v1"},
    )
    manifest = verify_pretrained_artifacts(tmpdir)
    settings_path = Path(tmpdir) / "generation_config.json"
    write_generation_settings(settings_path, GenerationSettings(max_new_tokens=8))
    print(manifest)
    print(settings_path.read_text())